In [1]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sionna.phy


import tensorflow as tf
# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

sionna.phy.config.seed = 42 # Set seed for reproducible results

In [2]:
from my_pusch_config import UeConfig, SystemConfig, MyConfig, MyPUSCHConfig

In [ ]:
import openvino as ov
from tensorflow.keras.layers import Layer, Conv2D, LayerNormalization, SeparableConv2D
from tensorflow.nn import relu

class ResidualBlock(tf.keras.Model):
    r"""
    This Keras layer implements a convolutional residual block made of two convolutional layers with ReLU activation, layer normalization, and a skip connection.
    The number of convolutional channels of the input must match the number of kernel of the convolutional layers ``num_conv_channel`` for the skip connection to work.

    Input
    ------
    : [batch size, num time samples, num subcarriers, num_conv_channel], tf.float
    Input of the layer

    Output
    -------
    : [batch size, num time samples, num subcarriers, num_conv_channel], tf.float
    Output of the layer
    """

    def build(self, input_shape):
        self._layer_norm_1 = LayerNormalization(axis=[-1,-2,-3])
        self._conv_1 = Conv2D(filters= 128,
            kernel_size=[3,3],
            padding='same',
            activation=None)

        self._layer_norm_2 = LayerNormalization(axis=[-1,-2,-3])
        self._conv_2 = Conv2D(filters= 128,
            kernel_size=[3,3],
            padding='same',
            activation=None)

    def call(self, inputs):
        z = self._layer_norm_1(inputs)
        z = relu(z)
        z = self._conv_1(z)
        z = self._layer_norm_2(z)
        z = relu(z)
        z = self._conv_2(z) # [batch size, num time samples, num subcarriers, num_channels]
        # Skip connection
        z = z + inputs

        return z

class CustomNeuralReceiver(tf.keras.Model):
    r"""
    Keras layer implementing a residual convolutional neural receiver.

    This neural receiver is fed with the post-DFT received samples, forming a resource grid of size num_of_symbols x fft_size, and computes LLRs on the transmitted coded bits.
    These LLRs can then be fed to an outer decoder to reconstruct the information bits.

    Input
    ------
    y_no: [batch size, num ofdm symbols, num subcarriers, 2*num rx antenna + 1], tf.float32
    Concatenated received samples and noise variance.
    (
    y : [batch size, num rx antenna, num ofdm symbols, num subcarriers], tf.complex
    Received post-DFT samples.

    no : [batch size], tf.float32
    Noise variance. At training, a different noise variance value is sampled for each batch example.
    )
    Output
    -------
    : [batch size, num ofdm symbols, num subcarriers, num_bits_per_symbol]
    LLRs on the transmitted bits.
    """

    def __init__(self, training = False):
        super(CustomNeuralReceiver, self).__init__()
        self._training = training

    def build(self, input_shape):

        # Input convolution
        self._input_conv = Conv2D(filters= 128,
            kernel_size=[3,3],
            padding='same',
            activation=None)
        # Residual blocks
        self._res_block_1 = ResidualBlock()
        self._res_block_2 = ResidualBlock()
        self._res_block_3 = ResidualBlock()
        self._res_block_4 = ResidualBlock()
        # Output conv
        self._output_conv = Conv2D(filters= 2, # QPSK
        kernel_size=[3,3],
        padding='same',
        activation=None)

    def call(self, inputs):
        # Input conv
      

        z = inputs
        z = self._input_conv(z)
        # Residual blocks
        z = self._res_block_1(z)
        z = self._res_block_2(z)
        z = self._res_block_3(z)
        z = self._res_block_4(z)
        # Output conv
        z = self._output_conv(z)

        return z
    
import pickle
def load_weights(model, pretrained_weights_path):
    # Build Model with random input
    # Load weights
    with open(pretrained_weights_path, 'rb') as f:
        weights = pickle.load(f)
        model.set_weights(weights)
        print(f"Loaded pretrained weights from {pretrained_weights_path}")

model = CustomNeuralReceiver(training = False)
inputs = tf.zeros([1,48,14,18])
model(inputs)
model.summary()

#load_weights(_model, '/content/drive/MyDrive/Pusch_data/Model_weights/model_weight_FULL_RB_epoch_40.pkl')
# load_weights(_model, '../model_weight_FULL_RB_epoch_40.pkl')
load_weights(model, '/workspaces/thanh/weight_4RB_batchsize_1024_186k_sample_dynamic_config_epoch120.pkl')

Model: "custom_neural_receiver_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_10 (Conv2D)          multiple                  20864     
                                                                 
 residual_block_4 (Residual  multiple                  639232    
 Block)                                                          
                                                                 
 residual_block_5 (Residual  multiple                  639232    
 Block)                                                          
                                                                 
 residual_block_6 (Residual  multiple                  639232    
 Block)                                                          
                                                                 
 residual_block_7 (Residual  multiple                  639232    
 Block)                                   

Loaded pretrained weights from /workspaces/thanh/weight_4RB_batchsize_1024_186k_sample_dynamic_config_epoch120.pkl


In [4]:
core = ov.Core()
compiled_model = core.compile_model("model.xml", "AUTO")

In [5]:
input_layer = compiled_model.input(0)
output_layer = compiled_model.output(0)

print(f"Input shape: {input_layer.shape}")
print(f"Output shape: {output_layer.shape}")

Input shape: [1,48,14,18]
Output shape: [1,48,14,2]


In [7]:
# 5. Perform inference
import numpy as np
input_data = np.zeros([1,48,14,18]).astype(np.float32)
results = compiled_model([input_data])[output_layer]
results

array([[[[-4.0055285 ,  2.1578388 ],
         [-0.45978776,  1.3798231 ],
         [ 0.20066252,  1.0616083 ],
         ...,
         [-0.83932686,  1.8955933 ],
         [ 0.06308392,  1.8673636 ],
         [ 1.4192811 ,  0.8539788 ]],

        [[-4.0288935 , -0.8593795 ],
         [-1.7379981 , -0.4301593 ],
         [ 1.9850994 , -3.3299096 ],
         ...,
         [ 0.57069683, -6.4908414 ],
         [ 0.37533486, -3.1370509 ],
         [ 3.0693262 , -3.0310695 ]],

        [[-5.6936116 , -1.2508421 ],
         [-1.199581  , -4.266326  ],
         [ 1.3283039 , -4.412203  ],
         ...,
         [-1.9604871 , -0.8441701 ],
         [ 2.6304383 , -3.3543293 ],
         [ 3.0618417 , -2.933988  ]],

        ...,

        [[-4.8890886 , -0.8794136 ],
         [ 0.517171  , -3.2053661 ],
         [ 2.6119618 , -3.78735   ],
         ...,
         [-8.299156  , -0.75559884],
         [ 0.8483558 , -5.744438  ],
         [ 3.421765  , -3.665254  ]],

        [[-2.3098145 , -2.1227157 